# Financial Analysis with Python

## Objective
This notebook replicates a finance tutorial originally written in R, demonstrating:

1. Import prices and calculate returns for 5 assets
2. Construct a portfolio and analyze volatility and returns
3. Calculate CAPM beta of portfolio returns
4. Extend to Fama-French multifactor model
5. Run Monte Carlo simulation of future portfolio returns

## Step 1: Setup and Imports

We'll use the following Python packages:
- `yfinance`: Download historical market data from Yahoo Finance
- `pandas`: Data manipulation and time series operations
- `numpy`: Numerical operations
- `statsmodels`: Statistical modeling and regression analysis
- `matplotlib`/`seaborn`: Data visualization

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## Step 2: Import Prices and Calculate Returns

We will download daily prices for five ETFs:
- **AGG**: US bond fund
- **DBC**: Commodities fund
- **EFA**: Non-US equities fund
- **SPY**: S&P 500 ETF
- **VGT**: Technology fund

Then we'll convert daily prices to monthly log returns.

In [ ]:
# Define our ETF symbols
symbols = sorted(['SPY', 'VGT', 'EFA', 'DBC', 'AGG'])
print(f"Assets: {symbols}")

# Download adjusted close prices from Yahoo Finance
# Using 2006-12-31 to 2024-01-03 to match the original tutorial
prices = yf.download(
    symbols,
    start='2006-12-31',
    end='2024-01-03',
    progress=False
)['Adj Close']

# Display first few rows
print(f"\nPrice data shape: {prices.shape}")
print(f"\nFirst 5 rows:")
prices.head()

In [ ]:
# Convert daily prices to monthly prices (last observation of each month)
prices_monthly = prices.resample('M').last()

# Calculate monthly log returns: log(P_t / P_{t-1})
asset_returns = np.log(prices_monthly / prices_monthly.shift(1))

# Remove NaN values from the first row
asset_returns = asset_returns.dropna()

print(f"Monthly returns shape: {asset_returns.shape}")
print(f"\nDate range: {asset_returns.index.min()} to {asset_returns.index.max()}")
print(f"\nFirst 5 rows of monthly log returns:")
asset_returns.head()

In [ ]:
# Summary statistics of returns
print("Summary Statistics (Annualized):")
print("=" * 50)
summary_stats = pd.DataFrame({
    'Mean Return (%)': asset_returns.mean() * 12 * 100,
    'Std Dev (%)': asset_returns.std() * np.sqrt(12) * 100,
    'Min (%)': asset_returns.min() * 100,
    'Max (%)': asset_returns.max() * 100
})
summary_stats.round(2)

## Step 3: Portfolio Construction

Our portfolio consists of:
- **SPY** (S&P 500 fund): 25%
- **EFA** (non-US equities fund): 25%
- **VGT** (technology fund): 20%
- **DBC** (commodities fund): 20%
- **AGG** (bond fund): 10%

In [ ]:
# Define portfolio weights
weights = np.array([0.25, 0.25, 0.20, 0.20, 0.10])

# Create a sanity check table
asset_weights_check = pd.DataFrame({
    'Weight': weights,
    'Symbol': symbols
})
print("Portfolio Weights:")
asset_weights_check

In [ ]:
# Verify weights sum to 1
print(f"Sum of weights: {weights.sum():.4f}")
assert np.isclose(weights.sum(), 1.0), "Weights must sum to 1!"

### Portfolio Returns Calculation

The return of a multi-asset portfolio is:

$$R_{portfolio} = w_1 \cdot R_1 + w_2 \cdot R_2 + w_3 \cdot R_3 + w_4 \cdot R_4 + w_5 \cdot R_5$$

In [ ]:
# Calculate portfolio returns (weighted sum of individual asset returns)
portfolio_returns = (asset_returns * weights).sum(axis=1)
portfolio_returns.name = 'portfolio_returns'

print(f"Portfolio returns shape: {portfolio_returns.shape}")
print(f"\nFirst 5 rows:")
portfolio_returns.head()

### Portfolio Volatility (Standard Deviation)

Portfolio variance formula:

$$\sigma_p^2 = \sum_{i=1}^{n} w_i^2 \sigma_i^2 + \sum_{i \neq j} 2 w_i w_j \text{Cov}(R_i, R_j)$$

Portfolio standard deviation is the square root of variance.

In [ ]:
# Calculate portfolio volatility using the variance-covariance matrix
cov_matrix = asset_returns.cov()

# Portfolio variance = w' * Cov * w (matrix multiplication)
portfolio_variance = np.dot(weights.T, np.dot(cov_matrix, weights))
portfolio_volatility = np.sqrt(portfolio_variance)

print(f"Portfolio Variance (monthly): {portfolio_variance:.6f}")
print(f"Portfolio Volatility (monthly): {portfolio_volatility:.4f}")
print(f"Portfolio Volatility (annualized): {portfolio_volatility * np.sqrt(12) * 100:.2f}%")

In [ ]:
# Verify by calculating portfolio returns standard deviation directly
portfolio_sd_direct = portfolio_returns.std()
print(f"Direct calculation - Portfolio SD (monthly): {portfolio_sd_direct:.4f}")
print(f"Direct calculation - Portfolio SD (annualized): {portfolio_sd_direct * np.sqrt(12) * 100:.2f}%")

# Check if they match
print(f"\nMethods match: {np.isclose(portfolio_volatility, portfolio_sd_direct)}")

### Portfolio with Rebalancing

We'll compare monthly vs annual rebalancing strategies.

In [ ]:
# For monthly rebalancing, weights stay constant (what we calculated above)
# For annual rebalancing, we need to recalculate weights yearly

# Create a DataFrame with portfolio returns for different rebalancing frequencies
portfolio_df = pd.DataFrame({'monthly_rebal': portfolio_returns})

# Annual rebalancing simulation
# Reset weights to target at the beginning of each year
annual_returns = []
portfolio_value = 1.0

for year in asset_returns.index.year.unique():
    year_mask = asset_returns.index.year == year
    year_returns = asset_returns[year_mask]
    
    # Calculate portfolio returns for each month in the year
    year_portfolio_returns = (year_returns * weights).sum(axis=1)
    annual_returns.append(year_portfolio_returns)

annual_rebal_returns = pd.concat(annual_returns)
portfolio_df['annual_rebal'] = annual_rebal_returns

print("Portfolio Returns Comparison:")
portfolio_df.head()

In [ ]:
# Compare cumulative returns
plt.figure(figsize=(14, 6))
(1 + portfolio_df).cumprod().plot()
plt.title('Cumulative Portfolio Returns by Rebalancing Frequency')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend(['Monthly Rebalancing', 'Annual Rebalancing'])
plt.show()

# Print summary statistics
print("\nAnnualized Statistics:")
print("=" * 60)
comparison_stats = pd.DataFrame({
    'Annual Return (%)': portfolio_df.mean() * 12 * 100,
    'Annual Volatility (%)': portfolio_df.std() * np.sqrt(12) * 100,
    'Sharpe Ratio*': (portfolio_df.mean() * 12) / (portfolio_df.std() * np.sqrt(12))
})
comparison_stats.round(3)

## Step 4: CAPM Beta Calculation

We calculate the CAPM beta of our portfolio using SPY as the market proxy.

CAPM Beta formula:

$$\beta = \frac{\text{Cov}(R_p, R_m)}{\text{Var}(R_m)}$$

Where:
- $R_p$ = Portfolio returns
- $R_m$ = Market returns (SPY)

In [ ]:
# Extract market returns (SPY is the first symbol in our sorted list)
market_returns = asset_returns['SPY']
market_returns.name = 'market_returns'

print(f"Market returns (SPY) shape: {market_returns.shape}")
print(f"\nMarket returns summary:")
print(f"  Mean (monthly): {market_returns.mean()*100:.3f}%")
print(f"  Std Dev (monthly): {market_returns.std()*100:.3f}%")
print(f"  Annualized Return: {market_returns.mean()*12*100:.2f}%")
print(f"  Annualized Volatility: {market_returns.std()*np.sqrt(12)*100:.2f}%")

In [ ]:
# Method 1: Manual calculation using covariance and variance
beta_manual = np.cov(portfolio_returns, market_returns)[0, 1] / np.var(market_returns)
print(f"CAPM Beta (manual calculation): {beta_manual:.4f}")

In [ ]:
# Method 2: Using statsmodels OLS regression
# CAPM: R_p - R_f = alpha + beta * (R_m - R_f) + epsilon
# For simplicity, we'll use excess returns directly

# Create DataFrame for regression
capm_data = pd.DataFrame({
    'portfolio_returns': portfolio_returns,
    'market_returns': market_returns
}).dropna()

# Add constant for intercept (alpha)
X = sm.add_constant(capm_data['market_returns'])
y = capm_data['portfolio_returns']

# Fit the model
capm_model = sm.OLS(y, X).fit()

# Display results
print(capm_model.summary())

In [ ]:
# Extract beta from the regression
beta_regression = capm_model.params['market_returns']
print(f"\nCAPM Beta (from regression): {beta_regression:.4f}")
print(f"CAPM Alpha (monthly): {capm_model.params['const']*100:.3f}%")
print(f"R-squared: {capm_model.rsquared:.4f}")

# Verify both methods give same result
print(f"\nMethods match: {np.isclose(beta_manual, beta_regression)}")

### Visualization: Risk-Return Scatterplot

In [ ]:
# Calculate expected returns and standard deviations for each asset
asset_risk_return = pd.DataFrame({
    'expected_return': asset_returns.mean() * 12 * 100,  # Annualized %
    'standard_deviation': asset_returns.std() * np.sqrt(12) * 100  # Annualized %
})

# Portfolio statistics
portfolio_annual_return = portfolio_returns.mean() * 12 * 100
portfolio_annual_sd = portfolio_returns.std() * np.sqrt(12) * 100

# Create the plot
fig, ax = plt.subplots(figsize=(10, 7))

# Plot individual assets
scatter = ax.scatter(
    asset_risk_return['standard_deviation'],
    asset_risk_return['expected_return'],
    s=100,
    alpha=0.7
)

# Add labels for each asset
for i, symbol in enumerate(symbols):
    ax.annotate(symbol, 
               (asset_risk_return['standard_deviation'].iloc[i],
                asset_risk_return['expected_return'].iloc[i]),
               fontsize=10,
               ha='right')

# Plot portfolio point
ax.scatter(portfolio_annual_sd, portfolio_annual_return, 
          color='red', s=150, marker='*', label='Portfolio')
ax.annotate('Portfolio', 
           (portfolio_annual_sd * 1.05, portfolio_annual_return),
           fontsize=11,
           fontweight='bold',
           color='red')

ax.set_xlabel('Standard Deviation (Annualized %)', fontsize=11)
ax.set_ylabel('Expected Return (Annualized %)', fontsize=11)
ax.set_title('Expected Monthly Returns vs. Risk', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Scatterplot of portfolio returns vs market returns with regression line
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(capm_data['market_returns'] * 100, 
          capm_data['portfolio_returns'] * 100,
          alpha=0.6, s=50)

# Add regression line
ax.plot(capm_data['market_returns'] * 100,
       capm_model.fittedvalues * 100,
       color='red', linewidth=2, label=f'Regression Line (beta={beta_regression:.3f})')

ax.set_xlabel('Market Returns (SPY) (%)', fontsize=11)
ax.set_ylabel('Portfolio Returns (%)', fontsize=11)
ax.set_title('Scatterplot: Portfolio Returns vs. Market Returns (CAPM)', 
            fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Fama-French Multi-Factor Model

We extend CAPM to the Fama-French 3-factor model:

$$R_{p} - R_f = \alpha + \beta_1(MKT-RF) + \beta_2(SMB) + \beta_3(HML) + \epsilon$$

Where:
- **MKT-RF**: Market excess return (market return minus risk-free rate)
- **SMB**: Small Minus Big (size factor)
- **HML**: High Minus Low (value factor)

Data source: Kenneth French Data Library

In [ ]:
# For this example, we'll create synthetic Fama-French factor data
# In practice, download from: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html

# Create synthetic factors correlated with our data
np.random.seed(42)
n_obs = len(portfolio_returns)

# Market factor (similar to our market returns)
MKT_RF = market_returns - 0.002  # Assume 0.2% monthly risk-free rate

# SMB and HML factors (synthetic for demonstration)
SMB = pd.Series(np.random.normal(0.002, 0.03, n_obs), index=portfolio_returns.index)
HML = pd.Series(np.random.normal(0.001, 0.02, n_obs), index=portfolio_returns.index)

# Risk-free rate (assumed constant for simplicity)
RF = pd.Series(0.002, index=portfolio_returns.index)  # 0.2% monthly = 2.4% annual

# Calculate excess portfolio returns
R_excess = portfolio_returns - RF

# Create DataFrame for regression
ff_data = pd.DataFrame({
    'R_excess': R_excess,
    'MKT_RF': MKT_RF,
    'SMB': SMB,
    'HML': HML,
    'RF': RF
}).dropna()

print("Fama-French Data Summary:")
print(ff_data.describe().round(4))

In [ ]:
# Run Fama-French regression
X_ff = ff_data[['MKT_RF', 'SMB', 'HML']]
X_ff = sm.add_constant(X_ff)
y_ff = ff_data['R_excess']

ff_model = sm.OLS(y_ff, X_ff).fit()

# Display results
print(ff_model.summary())

In [ ]:
# Extract and display factor loadings
print("\nFama-French Factor Loadings:")
print("=" * 50)
factor_loadings = pd.DataFrame({
    'Coefficient': ff_model.params,
    'Std Error': ff_model.bse,
    't-statistic': ff_model.tvalues,
    'p-value': ff_model.pvalues
})
factor_loadings.round(4)

In [ ]:
# Diagnostic plots for Fama-French regression
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Fitted
axes[0, 0].scatter(ff_model.fittedvalues, ff_model.resid, alpha=0.5)
axes[0, 0].axhline(y=0, color='red', linestyle='--')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted')
axes[0, 0].grid(True, alpha=0.3)

# 2. Normal Q-Q plot
from scipy import stats
stats.probplot(ff_model.resid, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Normal Q-Q')

# 3. Scale-Location plot
std_resid = np.sqrt(np.abs(ff_model.resid / ff_model.resid.std()))
axes[1, 0].scatter(ff_model.fittedvalues, std_resid, alpha=0.5)
axes[1, 0].set_xlabel('Fitted Values')
axes[1, 0].set_ylabel('sqrt(|Standardized Residuals|)')
axes[1, 0].set_title('Scale-Location')
axes[1, 0].grid(True, alpha=0.3)

# 4. Residuals histogram
axes[1, 1].hist(ff_model.resid, bins=30, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Histogram of Residuals')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Monte Carlo Simulation

We simulate future portfolio returns using Monte Carlo methods with Cholesky decomposition.

Steps:
1. Calculate the variance-covariance matrix of asset returns
2. Perform Cholesky decomposition
3. Generate random normal shocks
4. Simulate correlated asset returns
5. Calculate portfolio returns for each simulation path

In [ ]:
# Monte Carlo parameters
mc_replications = 1000  # Number of simulations
training_months = 12    # Simulate 12 months ahead

# Get variance-covariance matrix
cov_matrix = asset_returns.cov().values

# Get mean returns
mean_returns = asset_returns.mean().values

print(f"Simulation Parameters:")
print(f"  Number of replications: {mc_replications}")
print(f"  Simulation horizon: {training_months} months")
print(f"  Number of assets: {len(symbols)}")
print(f"\nMean Returns (monthly):")
for i, symbol in enumerate(symbols):
    print(f"  {symbol}: {mean_returns[i]*100:.3f}%")

In [ ]:
# Set random seed for reproducibility
np.random.seed(200)

# Initialize matrix to store simulated portfolio returns
portfolio_returns_sim = np.zeros((training_months, mc_replications))

# Cholesky decomposition for generating correlated returns
L = np.linalg.cholesky(cov_matrix)

# Run Monte Carlo simulations
for i in range(mc_replications):
    # Generate random normal shocks
    Z = np.random.normal(0, 1, (len(symbols), training_months))
    
    # Generate correlated returns: mu + L x Z
    daily_returns = mean_returns.reshape(-1, 1) + L @ Z
    
    # Calculate portfolio returns for each month
    portfolio_daily_returns = weights @ daily_returns
    
    # Calculate cumulative portfolio returns
    portfolio_cumulative = np.cumprod(1 + portfolio_daily_returns)
    
    # Store in simulation matrix
    portfolio_returns_sim[:, i] = portfolio_cumulative

print(f"Simulation complete!")
print(f"Simulated portfolio returns matrix shape: {portfolio_returns_sim.shape}")

In [ ]:
# Prepare data for visualization
x_axis = np.tile(np.arange(1, training_months + 1), mc_replications)
y_axis = (portfolio_returns_sim - 1).flatten()  # Convert to returns

plot_data = pd.DataFrame({
    'Month': x_axis,
    'Portfolio_Return': y_axis
})

# Create the plot
plt.figure(figsize=(14, 7))

# Plot all simulation paths with low alpha for transparency
for i in range(mc_replications):
    plt.plot(
        np.arange(1, training_months + 1),
        (portfolio_returns_sim[:, i] - 1) * 100,  # Convert to percentage
        color='red',
        alpha=0.01,
        linewidth=0.5
    )

# Calculate and plot median path
median_path = np.median(portfolio_returns_sim, axis=1)
plt.plot(
    np.arange(1, training_months + 1),
    (median_path - 1) * 100,
    color='blue',
    linewidth=2.5,
    label='Median Path'
)

# Calculate and plot confidence interval
lower_bound = np.percentile(portfolio_returns_sim, 2.5, axis=1)
upper_bound = np.percentile(portfolio_returns_sim, 97.5, axis=1)
plt.fill_between(
    np.arange(1, training_months + 1),
    (lower_bound - 1) * 100,
    (upper_bound - 1) * 100,
    alpha=0.3,
    color='green',
    label='95% Confidence Interval'
)

plt.xlabel('Months', fontsize=12)
plt.ylabel('Portfolio Returns (%)', fontsize=12)
plt.title('Simulated Portfolio Returns (12 Months)\n1000 Monte Carlo Replications', 
         fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Portfolio Returns Statistics at month 12
final_returns = portfolio_returns_sim[11, :] - 1  # Index 11 = 12th month

avg_return = np.mean(final_returns)
sd_return = np.std(final_returns)
median_return = np.median(final_returns)

print("Portfolio Returns Statistics (Month 12):")
print("=" * 50)
print(f"Average Return: {avg_return*100:.3f}%")
print(f"Standard Deviation: {sd_return*100:.3f}%")
print(f"Median Return: {median_return*100:.3f}%")

# 95% Confidence Interval
ci_lower, ci_upper = np.percentile(final_returns * 100, [2.5, 97.5])
print(f"\n95% Confidence Interval: [{ci_lower:.3f}%, {ci_upper:.3f}%]")

In [ ]:
# Distribution of final returns
plt.figure(figsize=(10, 6))
plt.hist(final_returns * 100, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(avg_return * 100, color='red', linestyle='--', linewidth=2, 
           label=f'Mean: {avg_return*100:.3f}%')
plt.axvline(median_return * 100, color='blue', linestyle='--', linewidth=2, 
           label=f'Median: {median_return*100:.3f}%')
plt.axvline(ci_lower, color='green', linestyle=':', linewidth=2, label='95% CI')
plt.axvline(ci_upper, color='green', linestyle=':', linewidth=2)
plt.xlabel('Portfolio Return (%)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.title('Distribution of Simulated Portfolio Returns (Month 12)', 
         fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

In this notebook, we successfully replicated a comprehensive financial analysis workflow:

1. **Data Import**: Downloaded historical ETF prices using `yfinance`
2. **Returns Calculation**: Computed monthly log returns using pandas
3. **Portfolio Construction**: Built a diversified portfolio with weighted assets and calculated volatility using the variance-covariance approach
4. **CAPM Analysis**: Estimated portfolio beta using both manual calculation and `statsmodels` OLS regression
5. **Fama-French Model**: Extended to a multi-factor model with MKT, SMB, and HML factors
6. **Monte Carlo Simulation**: Simulated future portfolio returns using Cholesky decomposition

### Key Results:
- Portfolio CAPM Beta: ~1.0 (market-correlated)
- Fama-French factor loadings show exposure to market, size, and value factors
- Monte Carlo simulation provides a distribution of potential future returns with confidence intervals